# Lab 6 — Transform, model & tune (Exam section 3 · 22%)

**How this lab works**
- Run cells top to bottom (`Shift+Enter`). After a kernel restart, run again from the top.
- Cells marked `# TODO` contain `<FILL-IN>`: replace it with your code, then run the `check(...)` cell.
- Stuck? Ask Claude in the terminal: *"hint for 1c"*. Solutions are in `notebooks/solutions/` (try first!).
- 🖥 **Local** means open-source Spark + Delta on your Mac. ☁ **Databricks** shows the real syntax for the exam.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # lets the notebook import labutils
from labutils import get_spark, path, reset_lab, show, checkpoint, table_path, landing
from labutils.check import check
from pyspark.sql import functions as F
spark = get_spark()
reset_lab("lab6")
spark.sql("""CREATE TABLE lab6.customers_bronze AS SELECT * FROM VALUES
  ('1', '  Ana Silva ', 'ana@x.com',  '2024-01-05', 'us', 'vip'),
  ('2', 'Ben Okoro',    NULL,         '2024-02-11', 'eu', NULL),
  (NULL,'Ghost Row',    'g@x.com',    '2024-03-01', 'us', 'bad id'),
  ('3', 'Chen Li',      'chen@x.com', '2024-03-20', 'EU', NULL),
  ('4', 'Dina Haddad',  NULL,         '2024-04-02', 'us', 'new')
  AS t(customer_id, full_name, email, signup_date, country, raw_note)""")
spark.sql("""CREATE TABLE lab6.orders AS SELECT * FROM VALUES
  (101, 1, 'US', 120.0), (102, 1, 'US', 80.0), (103, 2, 'EU', 45.5), (104, 3, 'EU', 300.0),
  (105, 9, 'US', 10.0),  (105, 9, 'US', 10.0), (106, 4, 'US', 99.9), (107, 3, 'US', 15.0)
  AS t(order_id, customer_id, country, amount)""")

## 6a · Clean bronze into silver

In [ ]:
# TODO 6a
bronze = spark.table("lab6.customers_bronze")
silver = (bronze
    .na.drop(subset=[<FILL-IN 1>])                          # drop rows with no customer_id
    .na.fill({<FILL-IN 2>})                                 # NULL email -> "unknown"
    .withColumn("customer_id", F.col("customer_id").cast(<FILL-IN 3>))   # STRING -> INT
    .withColumn("full_name", F.trim("full_name"))
    .withColumn("first_name", F.split("full_name", " ").getItem(0))
    .withColumn("last_name", F.split("full_name", " ").getItem(<FILL-IN 4>))
    .withColumn("country", F.upper("country"))
    .withColumn("signup_date", F.to_date("signup_date"))
    .withColumnRenamed("signup_date", <FILL-IN 5>)          # new name: "signed_up"
    .drop(<FILL-IN 6>))                                     # remove the raw_note column
silver.write.mode("overwrite").saveAsTable("lab6.customers_silver")
show(spark.table("lab6.customers_silver"))

In [ ]:
check("6a")

## 6b · Joins and unions

In [ ]:
# TODO 6b
from labutils.submit import submit
orders = spark.table("lab6.orders")
customers = spark.table("lab6.customers_silver")
inner_df = orders.join(customers.select("customer_id", "first_name"), "customer_id", <FILL-IN 1>)
left_df  = orders.join(customers.select("customer_id", "first_name"), "customer_id", <FILL-IN 2>)
multi_df = orders.join(customers.select("customer_id", "country"), <FILL-IN 3>, "inner")   # 2 keys
bcast_df = orders.join(<FILL-IN 4>(customers), "customer_id", "left")                     # broadcast hint
sizes  = spark.createDataFrame([("S",), ("M",), ("L",)], ["size"])
colors = spark.createDataFrame([("red",), ("blue",)], ["color"])
cross_df = sizes.<FILL-IN 5>(colors)                                                       # every combination
eu = spark.createDataFrame([(1, "EU"), (2, "EU")], ["id", "region"])
us = spark.createDataFrame([("US", 3), ("EU", 2)], ["region", "id"])                    # other column order!
union_all = eu.<FILL-IN 6>(us)                                                             # match by NAME
union_distinct = union_all.<FILL-IN 7>()                                                   # remove duplicates
for name, df in [("inner", inner_df), ("left", left_df), ("multi-key", multi_df), ("cross", cross_df),
                 ("union all", union_all), ("union distinct", union_distinct)]:
    print(f"{name:<15} rows = {df.count()}")
submit("6b", {"inner": inner_df.count(), "left": left_df.count(), "multi": multi_df.count(),
              "cross": cross_df.count(), "union_all": union_all.count(),
              "union_distinct": union_distinct.count(),
              "bcast_plan": bcast_df._jdf.queryExecution().executedPlan().toString()})

In [ ]:
check("6b")

## 6c · De-duplicate and aggregate

In [ ]:
# TODO 6c
orders = spark.table("lab6.orders")
clean = orders.<FILL-IN 1>(["order_id"])                       # remove the duplicate order 105
stats = (clean.groupBy("country")
    .agg(F.count("*").alias("n_orders"),
         <FILL-IN 2>("customer_id").alias("approx_customers"),   # fast approximate distinct count
         F.<FILL-IN 3>("amount").alias("avg_amount")))           # mean
stats.write.mode("overwrite").saveAsTable("lab6.country_stats")
show(stats)
show(clean.select("amount").<FILL-IN 4>())                      # count, mean, stddev, min, 25%, 50%, 75%, max

In [ ]:
check("6c")

## 6d · Basic Spark tuning

In [ ]:
# TODO 6d
from labutils.submit import submit
spark.conf.set("spark.sql.adaptive.enabled", "false")      # turn AQE off so you can see the raw plan
big = spark.range(100000).withColumn("k", F.col("id") % 100)
small = spark.range(100).withColumnRenamed("id", "k")
plan = lambda df: df._jdf.queryExecution().executedPlan().toString()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", <FILL-IN 1>)   # disable automatic broadcast
plan_off = plan(big.join(small, "k"))
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", <FILL-IN 2>)   # 10 MB again (in bytes, as text)
plan_on = plan(big.join(small, "k"))
spark.conf.set(<FILL-IN 3>, "8")                                      # number of partitions after a shuffle
parts = big.groupBy("k").count().rdd.getNumPartitions()

print("threshold -1  ->", "SortMergeJoin" if "SortMergeJoin" in plan_off else plan_off[:80])
print("threshold 10MB->", "BroadcastHashJoin" if "BroadcastHashJoin" in plan_on else plan_on[:80])
print("partitions after groupBy:", parts)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "2")
submit("6d", {"off": plan_off, "on": plan_on, "parts": parts})

In [ ]:
check("6d")